# TUAB Cross-Task Validation: ACBL + Gradient Isolation v1

Third dataset validation for NeuroState boundary detection.
Binary classification: normal (0) vs abnormal (1) EEG.
Uses BOTH train and eval TUAB splits for maximum data.
19 channels, 100 Hz, 30s epochs.

All model components included inline (self-contained).


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
import h5py
import json
from collections import defaultdict
from typing import Dict, Optional, Tuple
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    roc_auc_score, confusion_matrix, balanced_accuracy_score
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
# CONFIG

class Config:
    processed_dir = Path("data/processed")
    model_save_dir = Path("models")
    figures_dir = Path("figures/tuab_eval")

    task = "abnormal_detection"
    n_classes = 2
    n_channels = 19
    sfreq = 100
    epoch_duration = 30.0
    n_samples = int(sfreq * epoch_duration)  # 3000
    n_context = 3
    embed_dim = 128
    n_layers = 4
    dropout = 0.1
    contrast_scales = (1, 4, 16)
    cp_hidden = 64

    tokens_per_epoch = 196  # (3000 - 75) // 15 + 1
    total_tokens = tokens_per_epoch * n_context  # 588
    seconds_per_token = epoch_duration / tokens_per_epoch

    # Training
    batch_size = 32
    lr = 1e-4
    weight_decay = 1e-4
    n_train_epochs = 50
    warmup_epochs = 3
    formation_epochs = 12
    patience = 12
    acbl_weight = 0.3
    cls_weight = 1.0

    # Data loading
    max_train_chunks = 7  # To fit in memory
    seed = 42

Config.model_save_dir.mkdir(parents=True, exist_ok=True)
Config.figures_dir.mkdir(parents=True, exist_ok=True)

torch.manual_seed(Config.seed)
np.random.seed(Config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Config.seed)

In [3]:
# MODEL COMPONENTS 

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=19, n_samples=3000, embed_dim=128,
                 temporal_kernel=25, pool_kernel=75, pool_stride=15,
                 dropout=0.1):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, 40, (1, temporal_kernel),
                      padding=(0, temporal_kernel // 2)),
            nn.BatchNorm2d(40), nn.GELU())
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(40, 40, (n_channels, 1)),
            nn.BatchNorm2d(40), nn.GELU())
        self.pool = nn.AvgPool2d((1, pool_kernel), stride=(1, pool_stride))
        self.projection = nn.Sequential(
            nn.Conv2d(40, embed_dim, (1, 1)), nn.Dropout(dropout))
        self.seq_len = (n_samples - pool_kernel) // pool_stride + 1

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x)
        x = self.projection(x)
        return x.squeeze(2).permute(0, 2, 1)


class MultiResolutionEncoder(nn.Module):
    def __init__(self, n_channels=19, n_samples=3000,
                 embed_dim=128, dropout=0.1):
        super().__init__()
        self.enc_100 = PatchEmbedding(n_channels, n_samples, embed_dim,
                                       dropout=dropout)
        self.enc_50 = PatchEmbedding(n_channels, n_samples // 2, embed_dim,
                                      dropout=dropout)
        self.enc_25 = PatchEmbedding(n_channels, n_samples // 4, embed_dim,
                                      dropout=dropout)
        self.merge = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(),
            nn.Dropout(dropout))
        self.seq_len_100 = self.enc_100.seq_len

    def forward(self, x):
        e100 = self.enc_100(x)
        e50 = self.enc_50(x[:, :, ::2])
        e25 = self.enc_25(x[:, :, ::4])
        T = e100.shape[1]
        e50 = F.interpolate(e50.permute(0, 2, 1), size=T,
                            mode='linear', align_corners=False).permute(0, 2, 1)
        e25 = F.interpolate(e25.permute(0, 2, 1), size=T,
                            mode='linear', align_corners=False).permute(0, 2, 1)
        return self.merge(torch.cat([e100, e50, e25], dim=-1))


class ContrastiveBoundaryModule(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=64,
                 scales=(1, 4, 16), dropout=0.1):
        super().__init__()
        self.scales = scales
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
            for _ in scales])
        self.fusion = nn.Sequential(
            nn.Linear(len(scales), len(scales) * 2), nn.GELU(),
            nn.Linear(len(scales) * 2, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def _contrast(self, x, proj, offset):
        B, T, D = x.shape
        h = F.normalize(proj(x), dim=-1)
        if offset < T:
            h_shift = torch.roll(h, -offset, dims=1)
            h_shift[:, -offset:, :] = h[:, -offset:, :]
            sim = (h * h_shift).sum(dim=-1)
            c = 1.0 - (sim + 1.0) / 2.0
            c = torch.sigmoid(
                (c - 0.5) * self.temperature.abs().clamp(min=0.1))
        else:
            c = torch.zeros(B, T, device=x.device)
        return c

    def forward(self, x):
        per_scale = [self._contrast(x, p, o)
                     for p, o in zip(self.projections, self.scales)]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        consistency = sum(F.mse_loss(ps, fused.detach())
                         for ps in per_scale) / len(per_scale)
        return {'boundaries': fused, 'per_scale': per_scale,
                'boundary_loss': 0.01 * consistency}


class OriginalRegimeMask(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, boundaries):
        cum = torch.cumsum(boundaries, dim=1)
        same = torch.exp(-torch.abs(cum.unsqueeze(2) - cum.unsqueeze(1)))
        return same, 1.0 - same


class RegimeStructuredAttention(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, dropout=0.1, regime_mask_module=None):
        super().__init__()
        self.n_heads = n_intra + n_inter + n_cross
        self.head_dim = embed_dim // self.n_heads
        self.n_intra = n_intra
        self.n_inter = n_inter
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        self.regime_mask = regime_mask_module or OriginalRegimeMask()

    def forward(self, x, boundaries, return_attention=False):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        same, cross = self.regime_mask(boundaries)
        same = same.unsqueeze(1)
        cross = cross.unsqueeze(1)
        h1 = self.n_intra
        h2 = h1 + self.n_inter
        mask = torch.ones_like(attn)
        mask[:, :h1] = same.expand(B, self.n_intra, T, T)
        mask[:, h1:h2] = cross.expand(B, self.n_inter, T, T)
        attn = attn + torch.log(mask + 1e-6)
        w = F.softmax(attn, dim=-1)
        w = self.attn_drop(w)
        out = (w @ v).transpose(1, 2).reshape(B, T, D)
        out = self.proj_drop(self.out_proj(out))
        if return_attention:
            return out, w
        return out


class NeuroStateBlock(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, mlp_ratio=4.0, dropout=0.1,
                 regime_mask_module=None):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = RegimeStructuredAttention(
            embed_dim, n_intra, n_inter, n_cross, dropout,
            regime_mask_module)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout))

    def forward(self, x, boundaries, return_attention=False):
        if return_attention:
            attn_out, attn_w = self.attn(
                self.norm1(x), boundaries, return_attention=True)
            x = x + attn_out
            x = x + self.mlp(self.norm2(x))
            return x, attn_w
        x = x + self.attn(self.norm1(x), boundaries)
        x = x + self.mlp(self.norm2(x))
        return x


class MultiResContrastiveNeuroState(nn.Module):
    def __init__(self, n_channels=19, n_samples=3000, n_classes=2,
                 embed_dim=128, n_layers=4, dropout=0.1,
                 contrast_scales=(1, 4, 16), cp_hidden=64,
                 n_context_epochs=3):
        super().__init__()
        self.n_classes = n_classes
        self.n_context = n_context_epochs
        self.mr_encoder = MultiResolutionEncoder(
            n_channels, n_samples, embed_dim, dropout)
        self.tokens_per_epoch = self.mr_encoder.seq_len_100
        total_tokens = self.tokens_per_epoch * n_context_epochs
        self.pos_embed = nn.Parameter(
            torch.randn(1, total_tokens, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.epoch_embed = nn.Parameter(
            torch.randn(1, n_context_epochs, 1, embed_dim) * 0.02)
        self.changepoint_module = ContrastiveBoundaryModule(
            embed_dim, cp_hidden, contrast_scales, dropout)
        self.blocks = nn.ModuleList([
            NeuroStateBlock(embed_dim, dropout=dropout)
            for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim // 2, n_classes))
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"MultiResContrastiveNeuroState: {n_params/1e6:.2f}M params, "
              f"{total_tokens} tokens ({self.tokens_per_epoch}/epoch)")

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

In [4]:
# FORWARD WITH INTERMEDIATES

def forward_with_intermediates(model, x, detach_boundaries=False):
    B, N, C, T = x.shape
    epoch_embs = []
    for i in range(N):
        emb = model.mr_encoder(x[:, i])
        emb = emb + model.epoch_embed[:, i]
        epoch_embs.append(emb)
    full_seq = torch.cat(epoch_embs, dim=1)
    full_seq = model.pos_drop(full_seq + model.pos_embed)
    encoder_h = full_seq
    cp_out = model.changepoint_module(full_seq)
    boundaries = cp_out['boundaries']
    boundary_loss = cp_out['boundary_loss']
    boundaries_for_attn = boundaries.detach() if detach_boundaries else boundaries
    regime_attention = None
    for i, block in enumerate(model.blocks):
        if i == len(model.blocks) - 1:
            full_seq, attn_w = block(
                full_seq, boundaries_for_attn, return_attention=True)
            regime_attention = attn_w
        else:
            full_seq = block(full_seq, boundaries_for_attn)
    full_seq = model.norm(full_seq)
    tpe = model.tokens_per_epoch
    start = tpe * (N // 2)
    end = start + tpe
    pooled = full_seq[:, start:end, :].mean(dim=1)
    logits = model.head(pooled)
    return {
        'logits': logits, 'boundary_loss': boundary_loss,
        'boundaries': boundaries, 'encoder_h': encoder_h,
        'boundary_probs': boundaries, 'regime_attention': regime_attention,
    }

In [5]:
# ACBL LOSSES

class PseudoBoundaryLoss(nn.Module):
    def __init__(self, temperature=2.0, tokens_per_epoch=196, n_epochs=3):
        super().__init__()
        self.temperature = temperature
    def forward(self, boundary_probs, encoder_h):
        h_norm = F.normalize(encoder_h, dim=-1)
        sim = (h_norm[:, :-1] * h_norm[:, 1:]).sum(dim=-1)
        pseudo_targets = torch.sigmoid((1.0 - sim) * self.temperature)
        return F.binary_cross_entropy(
            boundary_probs[:, :-1], pseudo_targets.detach())


class AttentionPriorLoss(nn.Module):
    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 sigma=5.0, attn_prior_weight=0.5):
        super().__init__()
        self.tpe = tokens_per_epoch
        self.n_epochs = n_epochs
        self.sigma = sigma
        self.attn_prior_weight = attn_prior_weight

    def forward(self, boundary_probs, regime_attention, labels):
        B, T = boundary_probs.shape
        device = boundary_probs.device
        tpe = self.tpe
        targets = torch.zeros(B, T, device=device)
        mask = torch.zeros(B, device=device)
        for b in range(B):
            has_trans = False
            for ep in range(self.n_epochs - 1):
                if labels[b, ep] != labels[b, ep + 1]:
                    bt = (ep + 1) * tpe
                    pos = torch.arange(T, device=device, dtype=torch.float32)
                    g = torch.exp(-0.5 * ((pos - bt) / self.sigma) ** 2)
                    targets[b] = torch.max(targets[b], g)
                    has_trans = True
            if has_trans:
                mask[b] = 1.0
        if mask.sum() > 0:
            bnd_loss = F.binary_cross_entropy(
                boundary_probs, targets, reduction='none')
            bnd_loss = (bnd_loss.mean(dim=1) * mask).sum() / mask.sum()
        else:
            bnd_loss = torch.tensor(0.0, device=device)
        attn_loss = torch.tensor(0.0, device=device)
        if regime_attention is not None and mask.sum() > 0:
            attn = regime_attention.mean(dim=1) if regime_attention.dim() == 4 else regime_attention
            attn = attn / (attn.sum(dim=-1, keepdim=True) + 1e-8)
            prior = torch.zeros(B, T, T, device=device)
            for b in range(B):
                for ep in range(self.n_epochs):
                    s, e = ep * tpe, min((ep + 1) * tpe, T)
                    prior[b, s:e, s:e] = 1.0
            prior = prior / (prior.sum(dim=-1, keepdim=True) + 1e-8)
            kl = prior * (torch.log(prior.clamp(min=1e-8))
                          - torch.log(attn.clamp(min=1e-8)))
            attn_loss = (kl.sum(-1).mean(-1) * mask).sum() / mask.sum()
        return {
            'boundary_target_loss': bnd_loss,
            'attention_prior_loss': attn_loss,
            'total': bnd_loss + self.attn_prior_weight * attn_loss,
        }


class ACBLLoss(nn.Module):
    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 pseudo_weight=0.3, prior_weight=0.5,
                 pseudo_temperature=2.0, pseudo_temp_min=0.5,
                 pseudo_temp_anneal_epochs=15,
                 sigma=5.0, attn_prior_weight=0.5):
        super().__init__()
        self.pseudo_weight = pseudo_weight
        self.prior_weight = prior_weight
        self.pseudo_temp_min = pseudo_temp_min
        self.pseudo_temp_anneal_epochs = pseudo_temp_anneal_epochs
        self.pseudo_temperature_init = pseudo_temperature
        self.prong1 = PseudoBoundaryLoss(pseudo_temperature, tokens_per_epoch, n_epochs)
        self.prong2 = AttentionPriorLoss(tokens_per_epoch, n_epochs, sigma, attn_prior_weight)

    def anneal_temperature(self, epoch):
        if self.pseudo_temp_anneal_epochs <= 0:
            return
        progress = min(epoch / self.pseudo_temp_anneal_epochs, 1.0)
        self.prong1.temperature = self.pseudo_temperature_init - progress * (
            self.pseudo_temperature_init - self.pseudo_temp_min)

    def forward(self, boundary_probs, encoder_h, regime_attention,
                labels, epoch=0):
        self.anneal_temperature(epoch)
        pseudo_loss = self.prong1(boundary_probs, encoder_h)
        prior_results = self.prong2(boundary_probs, regime_attention, labels)
        total = (self.pseudo_weight * pseudo_loss
                 + self.prior_weight * prior_results['total'])
        return {
            'acbl_total': total,
            'pseudo_boundary_loss': pseudo_loss,
            'boundary_target_loss': prior_results['boundary_target_loss'],
            'attention_prior_loss': prior_results['attention_prior_loss'],
        }

In [6]:
# DATA LOADING — TUAB (train + eval combined)

def load_tuab_all(processed_dir, max_train_chunks=None):
    """Load both TUAB train chunks and eval into one combined array."""
    processed_dir = Path(processed_dir)
    all_epochs, all_labels, all_splits = [], [], []

    # Eval
    eval_path = processed_dir / 'tuab_eval_processed.h5'
    if eval_path.exists():
        print(f"Loading {eval_path.name}...")
        with h5py.File(eval_path, 'r') as f:
            ep = f['epochs'][:]
            lb = f['labels'][:]
        all_epochs.append(ep)
        all_labels.append(lb)
        all_splits.append(np.full(len(lb), 'eval', dtype='U5'))
        print(f"  Eval: {len(lb)} epochs")
    else:
        print(f"  Warning: {eval_path} not found")

    # Train chunks
    chunks = sorted(processed_dir.glob('tuab_train_chunk*.h5'))
    if max_train_chunks is not None:
        chunks = chunks[:max_train_chunks]
    if chunks:
        print(f"Loading {len(chunks)} train chunks...")
        for cp in tqdm(chunks, desc="Train chunks"):
            try:
                with h5py.File(cp, 'r') as f:
                    ep = f['epochs'][:]
                    lb = f['labels'][:]
                all_epochs.append(ep)
                all_labels.append(lb)
                all_splits.append(np.full(len(lb), 'train', dtype='U5'))
            except Exception as e:
                print(f"  Warning: {cp.name}: {e}")
    else:
        print("  No train chunks found")

    epochs = np.concatenate(all_epochs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    splits = np.concatenate(all_splits, axis=0)

    n_norm = np.sum(labels == 0)
    n_abn = np.sum(labels == 1)
    print(f"\nCombined: {len(labels)} epochs ({n_norm} normal, {n_abn} abnormal)")
    print(f"  From eval: {np.sum(splits == 'eval')}")
    print(f"  From train: {np.sum(splits == 'train')}")
    print(f"  Shape: {epochs.shape}")

    return epochs, labels, splits


class TUABWindowDataset(Dataset):
    """
    TUAB dataset with multi-epoch windows.
    Since TUAB has no subject IDs, we build consecutive windows
    from sequential epochs and do a random split.
    """
    def __init__(self, epochs, labels, indices, n_context=3):
        self.epochs = epochs
        self.labels = labels
        self.n_ctx = n_context
        self.windows = []

        # Build consecutive windows from the given indices
        # Sort indices to maintain temporal order
        sorted_idx = np.sort(indices)
        for i in range(len(sorted_idx) - n_context + 1):
            group = sorted_idx[i:i + n_context]
            # Check consecutive
            if np.all(np.diff(group) == 1):
                center = n_context // 2
                center_label = int(labels[group[center]])
                epoch_labels = [int(labels[g]) for g in group]
                self.windows.append({
                    'indices': group,
                    'label': center_label,
                    'epoch_labels': epoch_labels,
                })

        wl = [w['label'] for w in self.windows]
        print(f"  {len(self.windows)} windows "
              f"({sum(wl)} abnormal, {len(wl) - sum(wl)} normal)")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        x = np.stack([self.epochs[i] for i in w['indices']], axis=0)
        return {
            'x': torch.tensor(x, dtype=torch.float32),
            'label': torch.tensor(w['label'], dtype=torch.long),
            'epoch_labels': torch.tensor(w['epoch_labels'], dtype=torch.long),
        }


def get_class_weighted_sampler(dataset):
    labels = [w['label'] for w in dataset.windows]
    counts = np.bincount(labels)
    weights = 1.0 / counts
    sample_w = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_w, len(sample_w))

In [7]:
# TRAINING

def train_acbl_isolation(model, acbl_loss_fn, train_loader, val_loader,
                         config, device):
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=config.lr, weight_decay=config.weight_decay)
    train_labels = [w['label'] for w in train_loader.dataset.windows]
    n0 = sum(1 for l in train_labels if l == 0)
    n1 = sum(1 for l in train_labels if l == 1)
    cw = torch.tensor([len(train_labels) / (2*n0),
                        len(train_labels) / (2*n1)],
                       dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)

    best_val_loss = float('inf')
    wait = 0
    history = []

    for epoch in range(config.n_train_epochs):
        if epoch < config.warmup_epochs:
            phase = "warmup"
        elif epoch < config.warmup_epochs + config.formation_epochs:
            phase = "form"
        else:
            phase = "full"
        use_acbl = epoch >= config.warmup_epochs
        detach_bnd = (phase == "form")

        if epoch == config.warmup_epochs + config.formation_epochs:
            wait = 0
            best_val_loss = float('inf')
            print("  >> Full phase: boundaries reconnected, patience reset")

        model.train()
        stats = defaultdict(list)
        for batch in tqdm(train_loader, desc=f'Ep{epoch:2d} [{phase:7s}]',
                          leave=False):
            x = batch['x'].to(device)
            y = batch['label'].to(device)
            ep_labels = batch['epoch_labels'].to(device)
            optimizer.zero_grad()
            out = forward_with_intermediates(model, x, detach_bnd)
            cls_loss = criterion(out['logits'], y) * config.cls_weight
            total_loss = cls_loss
            if use_acbl:
                acbl_out = acbl_loss_fn(
                    out['boundary_probs'], out['encoder_h'],
                    out['regime_attention'], ep_labels, epoch)
                total_loss = total_loss + config.acbl_weight * acbl_out['acbl_total']
                stats['acbl'].append(acbl_out['acbl_total'].item())
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            stats['cls'].append(cls_loss.item())
            with torch.no_grad():
                bnd = out['boundaries']
                stats['bnd_mean'].append(bnd.mean().item())
                stats['bnd_std'].append(bnd.std().item())

        # Validate
        model.eval()
        val_losses = []
        all_preds, all_labels_list = [], []
        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(device)
                y = batch['label'].to(device)
                out = forward_with_intermediates(model, x, False)
                val_losses.append(criterion(out['logits'], y).item())
                all_preds.extend(out['logits'].argmax(1).cpu().numpy())
                all_labels_list.extend(y.cpu().numpy())

        val_cls = np.mean(val_losses)
        val_acc = accuracy_score(all_labels_list, all_preds)
        log = {
            'epoch': epoch, 'phase': phase,
            'train_cls': np.mean(stats['cls']),
            'val_cls': val_cls, 'val_acc': val_acc,
            'bnd_mean': np.mean(stats['bnd_mean']),
            'bnd_std': np.mean(stats['bnd_std']),
        }
        history.append(log)
        print(f"  Ep {epoch:2d} [{phase:7s}] "
              f"cls={log['train_cls']:.4f} val={val_cls:.4f} "
              f"acc={val_acc:.3f} "
              f"bnd_m={log['bnd_mean']:.4f} bnd_s={log['bnd_std']:.4f}")

        if val_cls < best_val_loss:
            best_val_loss = val_cls
            wait = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'epoch': epoch, 'val_loss': val_cls, 'val_acc': val_acc,
            }, config.model_save_dir / 'acbl_isolation_tuab_best.pt')
        else:
            # Only allow early stopping after full phase begins
            if epoch >= config.warmup_epochs + config.formation_epochs:
                wait += 1
                if wait >= config.patience:
                    print(f"  Early stopping at epoch {epoch}")
                    break
    return history

In [8]:
# EVALUATION

def evaluate_tuab(model, test_loader, config, device):
    model.eval()
    results = {'per_window': [], 'classification': {
        'preds': [], 'labels': [], 'probs': []}}

    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating'):
            x = batch['x'].to(device)
            labels = batch['label']
            ep_labels = batch['epoch_labels']
            out = forward_with_intermediates(model, x, False)
            boundaries = out['boundaries'].cpu().numpy()
            probs = F.softmax(out['logits'], dim=1).cpu().numpy()
            preds = out['logits'].argmax(1).cpu().numpy()

            results['classification']['preds'].extend(preds.tolist())
            results['classification']['labels'].extend(labels.numpy().tolist())
            results['classification']['probs'].extend(probs[:, 1].tolist())

            for i in range(len(labels)):
                bnd = boundaries[i]
                info = {
                    'label': int(labels[i]),
                    'pred': int(preds[i]),
                    'prob_abnormal': float(probs[i, 1]),
                    'boundary_mean': float(bnd.mean()),
                    'boundary_std': float(bnd.std()),
                    'boundary_max': float(bnd.max()),
                    'epoch_labels': ep_labels[i].tolist(),
                }
                # Per-class boundary analysis
                el = ep_labels[i].numpy()
                tpe = config.tokens_per_epoch
                center_bnd = bnd[tpe:2*tpe]
                surround_bnd = np.concatenate([bnd[:tpe], bnd[2*tpe:]])
                info['center_bnd_mean'] = float(center_bnd.mean())
                info['surround_bnd_mean'] = float(surround_bnd.mean())
                info['center_vs_surround'] = float(
                    center_bnd.mean() - surround_bnd.mean())
                results['per_window'].append(info)
    return results


def compute_tuab_metrics(eval_results):
    all_w = eval_results['per_window']
    abn_w = [w for w in all_w if w['label'] == 1]
    norm_w = [w for w in all_w if w['label'] == 0]

    print(f"\n{'='*60}")
    print("TUAB BOUNDARY EVALUATION")
    print(f"{'='*60}")
    print(f"Windows: {len(abn_w)} abnormal, {len(norm_w)} normal")

    # Boundary stats
    abn_means = [w['boundary_mean'] for w in abn_w]
    norm_means = [w['boundary_mean'] for w in norm_w]
    abn_stds = [w['boundary_std'] for w in abn_w]
    norm_stds = [w['boundary_std'] for w in norm_w]
    sep = np.mean(abn_means) - np.mean(norm_means) if abn_means and norm_means else 0

    print(f"\n--- Boundary Activation ---")
    print(f"  Abnormal: mean={np.mean(abn_means):.4f}, std={np.mean(abn_stds):.4f}")
    print(f"  Normal:   mean={np.mean(norm_means):.4f}, std={np.mean(norm_stds):.4f}")
    print(f"  Separation: {sep:.4f}")

    # Center vs surround
    abn_deltas = [w['center_vs_surround'] for w in abn_w]
    norm_deltas = [w['center_vs_surround'] for w in norm_w]
    print(f"\n--- Center vs Surround ---")
    if abn_deltas:
        print(f"  Abnormal: delta={np.mean(abn_deltas):.4f}, "
              f"positive={sum(1 for d in abn_deltas if d > 0)}/{len(abn_deltas)}")
    if norm_deltas:
        print(f"  Normal:   delta={np.mean(norm_deltas):.4f}, "
              f"positive={sum(1 for d in norm_deltas if d > 0)}/{len(norm_deltas)}")

    # Classification
    cls = eval_results['classification']
    y_true = np.array(cls['labels'])
    y_pred = np.array(cls['preds'])
    y_prob = np.array(cls['probs'])

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auroc = float('nan')

    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        tn = fp = fn = tp = 0
        sens = spec = 0

    print(f"\n--- Classification ---")
    print(f"  Acc={acc:.4f} BalAcc={bal_acc:.4f} F1M={f1m:.4f}")
    print(f"  Kappa={kappa:.4f} AUROC={auroc:.4f}")
    print(f"  Sens={sens:.4f} Spec={spec:.4f}")
    print(f"  CM: TN={tn} FP={fp} FN={fn} TP={tp}")

    return {
        'boundary_separation': sep,
        'acc': acc, 'bal_acc': bal_acc, 'f1m': f1m,
        'kappa': kappa, 'auroc': auroc,
        'sensitivity': sens, 'specificity': spec,
        'bnd_mean_abnormal': np.mean(abn_means) if abn_means else 0,
        'bnd_mean_normal': np.mean(norm_means) if norm_means else 0,
        'bnd_std_abnormal': np.mean(abn_stds) if abn_stds else 0,
        'bnd_std_normal': np.mean(norm_stds) if norm_stds else 0,
    }

In [9]:
# MAIN

def main():
    print("=" * 60)
    print("TUAB CROSS-TASK VALIDATION: ACBL + ISOLATION v1")
    print("=" * 60)

    # Load all data (train + eval)
    epochs, labels, splits = load_tuab_all(
        Config.processed_dir, Config.max_train_chunks)

    n_channels = epochs.shape[1]
    print(f"\nData channels: {n_channels}, Config expects: {Config.n_channels}")

    # Pad or truncate channels to match config
    if n_channels < Config.n_channels:
        pad = np.zeros((len(epochs), Config.n_channels - n_channels,
                        epochs.shape[2]), dtype=epochs.dtype)
        epochs = np.concatenate([epochs, pad], axis=1)
        print(f"  Padded {n_channels} -> {Config.n_channels} channels")
    elif n_channels > Config.n_channels:
        epochs = epochs[:, :Config.n_channels, :]
        print(f"  Truncated {n_channels} -> {Config.n_channels} channels")

    # Random split: 70/15/15
    n = len(labels)
    rng = np.random.RandomState(Config.seed)
    idx = rng.permutation(n)
    n_train = int(0.60 * n)
    n_val = int(0.20 * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]

    print(f"\nSplit: {len(train_idx)} train, {len(val_idx)} val, "
          f"{len(test_idx)} test")
    for name, idx_set in [('Train', train_idx), ('Val', val_idx),
                           ('Test', test_idx)]:
        dist = dict(zip(*np.unique(labels[idx_set], return_counts=True)))
        print(f"  {name}: {len(idx_set)} epochs, {dist}")

    # Build windowed datasets
    print("\nBuilding datasets:")
    train_ds = TUABWindowDataset(epochs, labels, train_idx, Config.n_context)
    val_ds = TUABWindowDataset(epochs, labels, val_idx, Config.n_context)
    test_ds = TUABWindowDataset(epochs, labels, test_idx, Config.n_context)

    train_sampler = get_class_weighted_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=Config.batch_size,
                              sampler=train_sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=Config.batch_size,
                            shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=Config.batch_size,
                             shuffle=False, num_workers=0)

    # Model
    model = MultiResContrastiveNeuroState(
        n_channels=Config.n_channels,
        n_samples=Config.n_samples,
        n_classes=Config.n_classes,
        embed_dim=Config.embed_dim,
        n_layers=Config.n_layers,
        dropout=Config.dropout,
        contrast_scales=Config.contrast_scales,
        cp_hidden=Config.cp_hidden,
        n_context_epochs=Config.n_context,
    ).to(device)

    print(f"\nVerified: tokens_per_epoch = {model.tokens_per_epoch}")

    acbl_loss_fn = ACBLLoss(
        tokens_per_epoch=model.tokens_per_epoch,
        n_epochs=Config.n_context,
    )

    # Train
    print("\nTraining ACBL + Isolation v1 on TUAB:")
    history = train_acbl_isolation(model, acbl_loss_fn, train_loader,
                                   val_loader, Config, device)

    # Load best
    ckpt = torch.load(Config.model_save_dir / 'acbl_isolation_tuab_best.pt',
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"\nLoaded best from epoch {ckpt['epoch']} "
          f"(val_acc={ckpt['val_acc']:.4f})")

    # Evaluate
    print("\nBoundary evaluation on test set:")
    eval_results = evaluate_tuab(model, test_loader, Config, device)
    metrics = compute_tuab_metrics(eval_results)

    # Save
    save_path = Config.model_save_dir / 'tuab_eval_results.json'
    with open(save_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\nResults saved to {save_path}")

    hist_path = Config.model_save_dir / 'acbl_tuab_history.json'
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"History saved to {hist_path}")


if __name__ == "__main__":
    main()

TUAB CROSS-TASK VALIDATION: ACBL + ISOLATION v1
Loading tuab_eval_processed.h5...
  Eval: 12241 epochs
Loading 7 train chunks...
Train chunks: 100%|██████████| 7/7 [03:26<00:00, 29.55s/it]

Combined: 74395 epochs (67490 normal, 6905 abnormal)
  From eval: 12241
  From train: 62154
  Shape: (74395, 19, 3000)

Data channels: 19, Config expects: 19

Split: 44637 train, 14879 val, 14879 test
  Train: 44637 epochs, {0: 40562, 1: 4075}
  Val: 14879 epochs, {0: 13489, 1: 1390}
  Test: 14879 epochs, {0: 13439, 1: 1440}

Building datasets:
  16129 windows (1409 abnormal, 14720 normal)
  604 windows (52 abnormal, 552 normal)
  604 windows (65 abnormal, 539 normal)
MultiResContrastiveNeuroState: 1.07M params, 588 tokens (196/epoch)

Verified: tokens_per_epoch = 196

Training ACBL + Isolation v1 on TUAB:
  Ep  0 [warmup ] cls=0.2934 val=0.9162 acc=0.666 bnd_m=0.4992 bnd_s=0.0000
  Ep  1 [warmup ] cls=0.1663 val=0.9033 acc=0.742 bnd_m=0.4997 bnd_s=0.0000
  Ep  2 [warmup ] cls=0.1472 val=0.5578 acc=

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>